# Meme Explainer

Upload a meme image and get an AI-powered explanation of the humor behind it.

The pipeline:
1. **BLIP** (HuggingFace vision pipeline) generates a caption describing the image
2. **GPT-4o-mini** (OpenAI) interprets the caption and explains the joke
3. **Gradio** provides an interactive UI for uploading and viewing results

**Week 3 concepts used:** Google Colab (Day 1) · HuggingFace Pipelines (Day 2) · Tokenizers (Day 3) · Model Inference (Day 4) · Multi-step App with Gradio (Day 5)

In [7]:
%pip install -q transformers torch openai gradio Pillow python-dotenv

/Users/anthony/workspace/ai/llm_engineering/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import torch
from dotenv import load_dotenv
from transformers import pipeline
from openai import OpenAI
from PIL import Image
import gradio as gr

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

# Fall back to Google Colab secrets if not found in .env
if not openai_api_key:
    try:
        from google.colab import userdata
        openai_api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set — set it in .env or Colab Secrets")

DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'CUDA' if DEVICE == 0 else 'CPU'}")


In [ ]:
captioner = pipeline(
    "image-to-text",
    model="Salesforce/blip-image-captioning-large",
    device=DEVICE,
)
print("BLIP captioning model loaded")

openai_client = OpenAI(api_key=openai_api_key)
MODEL = "gpt-4o-mini"

## How It Works

Memes are tricky for AI because humor depends on cultural context, visual cues, and
implied meaning that a simple image classifier would miss.

Our approach chains two models:
- **BLIP** (`Salesforce/blip-image-captioning-large`) is a vision-language model that
  generates a text caption from the image. This bridges the visual and text domains.
- **GPT-4o-mini** takes that caption and applies its broad cultural knowledge to
  explain why the image is funny, what meme format it likely represents, and the
  underlying joke structure.

In [ ]:
SYSTEM_PROMPT = """You are a meme expert and humor analyst.

You will receive a caption generated by an image-captioning AI that describes a meme image.
Your job is to:

1. **Identify the meme format** — name it if it's a well-known template (e.g. "Distracted Boyfriend",
   "Drake Hotline Bling", "Two Buttons", etc.), or describe the visual format if unknown.
2. **Explain the joke** — what is funny about this image and why people share it.
3. **Cultural context** — briefly note where this meme comes from or when it was popular, if applicable.
4. **Humor rating** — rate the meme 1-10 and say what makes it land (or not).

Keep the tone fun and conversational. If the caption is ambiguous or unclear, make your best guess
and note the uncertainty."""


def explain_meme(image, progress=gr.Progress()):
    """Pipeline: image -> BLIP caption -> GPT explanation."""
    if image is None:
        return "No caption generated.", "Please upload a meme image first."

    progress(0.1, desc="Analyzing image with BLIP...")
    captions = captioner(image)
    caption = captions[0]["generated_text"]

    progress(0.5, desc="Asking GPT to explain the humor...")
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Here is what the image-captioning AI sees in this meme:\n\n\"{caption}\"\n\nExplain this meme."},
        ],
        temperature=0.7,
        max_tokens=500,
    )
    explanation = response.choices[0].message.content

    progress(1.0, desc="Done!")
    return caption, explanation

In [ ]:
with gr.Blocks(title="Meme Explainer") as ui:
    gr.Markdown("# Meme Explainer")
    gr.Markdown(
        "Upload a meme and let AI explain the joke. "
        "Uses **BLIP** for image captioning and **GPT-4o-mini** for humor analysis."
    )

    with gr.Row():
        image_input = gr.Image(type="pil", label="Upload a Meme")
        with gr.Column():
            caption_output = gr.Textbox(label="BLIP Caption", interactive=False)
            explanation_output = gr.Markdown(label="Explanation")

    explain_btn = gr.Button("Explain This Meme", variant="primary")

    explain_btn.click(
        fn=explain_meme,
        inputs=[image_input],
        outputs=[caption_output, explanation_output],
    )

ui.launch(inbrowser=True)